# 14.6 · LSTM / GRU 时序预测 / LSTM/GRU for Time Series

> **课程定位 / Where this fits**
> 第 6 课，**Part 14 · 时间序列**。从经典统计转向**深度学习**预测。
> Lesson 6, **Part 14 · Time Series**. From classical statistics to **deep-learning** forecasting.
>
> 前面的 ARIMA/Prophet 是统计方法。**LSTM/GRU**(12.1 学过)能学**复杂的非线性时序模式**, 还能轻松处理**多变量、长依赖**, 是深度时序预测的主力。关键的工程套路是**滑动窗口**: 把序列切成"用过去 N 步预测下一步"的样本, 喂给 LSTM。本课从零搭 LSTM 预测器, 并**诚实对比它和经典方法**——你会看到一个重要现实: **在小数据上, 深度学习往往打不过简单的经典方法**(DL 的优势要在大数据/多序列时才显现)。还会强调**缩放只在训练集拟合、按时间划分**等防泄漏要点。
> ARIMA/Prophet are statistical. **LSTM/GRU** (from 12.1) learn **complex nonlinear patterns** and handle **multivariate, long dependencies** easily — the workhorse of deep forecasting. The key engineering trick is the **sliding window**: slice the series into "predict the next step from the past N" samples for the LSTM. We build an LSTM forecaster from scratch and **honestly compare it to classical methods** — revealing an important reality: **on small data, deep learning often loses to simple classical methods** (DL's edge needs lots of data/many series). We also stress leakage-avoidance: **fit scaling on train only, split chronologically**.
>
> 💼 **实战/面试视角**："滑动窗口怎么构造 / 缩放为什么只在训练集fit / 单步vs多步(递归)预测 / 什么时候DL才值得用" 是深度时序常考。
> 💼 **Practical/interview angle:** "windowing / why fit scaling on train only / one-step vs recursive multi-step / when DL is worth it" — common.

> 📐 **符号约定 / Notation**
> - 窗口长度 $L$ —— 用过去多少步做输入 / lookback window length
> - 递归预测 —— 把预测值当输入再预测下一个 / recursive: feed predictions back

> 💡 **面试相关 / Interview-relevant**
> - "时序怎么做成监督学习(滑动窗口)"（出镜率 ★★★★★）
> - "数据缩放为什么只能在训练集上fit"（★★★★★，防泄漏）
> - "多步预测: 递归 vs 直接"（★★★★）
> - "什么时候用深度学习而非ARIMA"（★★★★★，数据量是关键）

---

## 学习目标 / Learning Objectives
1. 会用**滑动窗口**把时序变成监督学习问题。
   Turn a series into supervised learning with sliding windows.
2. 掌握**缩放只在训练集fit**、按时间划分等防泄漏要点。
   Master leakage-avoidance: scaling fit on train only, chronological split.
3. **从零搭 LSTM** 预测器并做递归多步预测。
   Build an LSTM forecaster from scratch with recursive multi-step forecast.
4. **诚实对比** LSTM 与经典方法, 理解何时该用 DL。
   Honestly compare LSTM to classical; understand when to use DL.

## 目录 / TOC
1. [深度学习做时序:滑动窗口 ⭐](#1)
2. [缩放与防泄漏 ⭐](#2)
3. [搭 LSTM 预测 + 递归多步 ⭐](#3)
4. [诚实对比经典方法 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 深度学习做时序:滑动窗口 ⭐ / Deep Learning for TS: Sliding Windows

神经网络要的是"输入特征 → 输出标签"的监督学习样本, 但时间序列是一长串数。**滑动窗口(sliding window)** 是把二者连接起来的关键(面试必会)：
Neural nets want supervised "features → label" samples, but a time series is one long sequence. The **sliding window** bridges them (must-know):
- 用**过去 $L$ 步**作为输入特征, **下一步**作为标签。
  Use the **past $L$ steps** as input features and the **next step** as the label.
- 在序列上**滑动**这个窗口, 生成大量 (输入窗口, 下一个值) 训练样本。
  **Slide** this window over the series to generate many (input window, next value) samples.

例如序列 `[a,b,c,d,e]`, 窗口 $L=2$: 样本 `([a,b]→c), ([b,c]→d), ([c,d]→e)`。这叫 **seq-to-one(序列到一个值)**。LSTM 读入窗口序列, 输出下一步预测。
E.g. for `[a,b,c,d,e]` with $L=2$: samples `([a,b]→c), ([b,c]→d), ([c,d]→e)`. This is **seq-to-one**. The LSTM reads the window and outputs the next-step prediction.

窗口长度 $L$ 要覆盖关心的依赖(有年度季节就至少 $L\ge12$)。下面可视化窗口化。
The window $L$ should cover the dependencies you care about (≥12 for yearly seasonality). Let's visualize windowing.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, time
import torch, torch.nn as nn
import warnings; warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid"); torch.manual_seed(0)
ap = [112,118,132,129,121,135,148,148,136,119,104,118,115,126,141,135,125,149,170,170,158,133,114,140,145,150,178,163,172,178,199,199,184,162,146,166,171,180,193,181,183,218,230,242,209,191,172,194,196,196,236,235,229,243,264,272,237,211,180,201,204,188,235,227,234,264,302,293,259,229,203,229,242,233,267,269,270,315,364,347,312,274,237,278,284,277,317,313,318,374,413,405,355,306,271,306,315,301,356,348,355,422,465,467,404,347,305,336,340,318,362,348,363,435,491,505,404,359,310,337,360,342,406,396,420,472,548,559,463,407,362,405,417,391,419,461,472,535,622,606,508,461,390,432]
ts = np.array(ap, dtype=float)
idx = pd.date_range("1949-01", periods=len(ts), freq="MS")

# 可视化滑动窗口: 用过去L步预测下一步 / visualize sliding window
L = 12
fig, ax = plt.subplots(figsize=(11, 3.6)); ax.plot(ts, alpha=0.4, color="gray")
for start in [0, 30, 60]:
    ax.plot(range(start, start+L), ts[start:start+L], "o-", label=f"输入窗口[{start}:{start+L}]")
    ax.plot(start+L, ts[start+L], "r*", ms=15)            # 要预测的下一步 / the next step to predict
ax.legend(); ax.set_title(f"滑动窗口(L={L}): 用过去{L}步(圆点)预测下一步(红星); 滑动生成大量样本")
plt.tight_layout(); plt.show()
print(f"滑动窗口把时序变成监督学习: 过去{L}步→下一步; 在序列上滑动 → 一堆(窗口,标签)样本(seq-to-one)")
print("窗口L要覆盖关心的依赖(有年度季节→L≥12)")


<a id="2"></a>
## 2. 缩放与防泄漏 ⭐ / Scaling & Avoiding Leakage

深度学习对**输入尺度敏感**, 时序通常要先**缩放**(如 MinMax 到 [0,1] 或标准化)。但这里藏着时序最容易犯的**泄漏陷阱**(面试高频)：
Deep learning is **scale-sensitive**, so series are usually **scaled** (MinMax to [0,1] or standardized) first. But here lurks the most common TS **leakage trap** (high-frequency):

> ⚠️ **缩放器只能在训练集上 `fit`**, 再用它 `transform` 训练和测试。**绝不能在整条序列(含测试集)上 fit**——否则测试集的最大/最小值/均值就"泄漏"进了缩放参数, 模型间接看到了未来, 评估虚高。这是和"不能打乱""按时间切"一脉相承的铁律。
> ⚠️ **Fit the scaler on the training set only**, then transform both. **Never fit on the whole series (including test)** — else the test set's min/max/mean leak into the scaling parameters, the model indirectly sees the future, and evaluation is inflated. Same family of rules as "no shuffling / chronological split."

同理, 滑动窗口、特征工程里的任何统计量(均值、标准差等)都**只能用过去/训练的数据**算。时序里"未来信息泄漏"无处不在, 是实战翻车的头号原因。
Likewise, any statistic in windowing/feature engineering (means, stds) must be computed **only from past/training data**. Future leakage is everywhere in TS and the #1 cause of real-world failures.


In [ ]:
train, test = ts[:120], ts[120:]                          # 按时间切: 前10年训练, 后2年测试 / chronological split
# ✅ 正确: 缩放器只用训练集的 min/max / CORRECT: fit scaler on TRAIN ONLY
mn, mx = train.min(), train.max()
def scale(x): return (x - mn) / (mx - mn)
def unscale(x): return x * (mx - mn) + mn
print(f"缩放参数只来自训练集: min={mn}, max={mx} (绝不用测试集的值!)")
print(f"  ✅ 正确做法: 用训练集的min/max缩放训练和测试")
print(f"  ❌ 错误做法: 用整条序列(含测试)的min/max → 测试集信息泄漏 → 评估虚高")

# 构造训练窗口 / build training windows
s_train = scale(train)
X, Y = [], []
for i in range(len(s_train) - L):
    X.append(s_train[i:i+L]); Y.append(s_train[i+L])      # 过去L步 → 下一步 / past L → next
X = torch.tensor(np.array(X), dtype=torch.float32).unsqueeze(-1)   # (样本, L, 1特征) / (N, L, 1)
Y = torch.tensor(np.array(Y), dtype=torch.float32).unsqueeze(-1)
print(f"\n训练窗口: X 形状 {tuple(X.shape)} = (样本数, 窗口长L, 特征数), Y 形状 {tuple(Y.shape)}")


<a id="3"></a>
## 3. 搭 LSTM 预测 + 递归多步 ⭐ / LSTM Forecaster + Recursive Multi-Step

搭一个 LSTM: 读入窗口序列, 输出下一步预测。训练好后做**多步预测**——预测第 25 个月需要第 13-24 月, 但测试期我们没有真实值, 所以用**递归预测**: 把模型自己的预测当作输入, 一步步往后推。
Build an LSTM: read the window, output the next step. For **multi-step forecasting** — predicting month 25 needs months 13–24, but in the test period we lack truths, so we use **recursive forecasting**: feed the model's own predictions back as inputs, stepping forward.

> ⚠️ **递归预测的坑(面试)**: 误差会**累积**——第一步的小错会成为第二步的输入, 越往后越偏。这是多步预测普遍变难的原因。另一种是**直接预测**(直接训练模型一次输出多步), 各有取舍。
> ⚠️ **Recursive pitfall:** errors **compound** — a small first-step error becomes the next input, drifting worse over time. Why multi-step is generally harder. The alternative is **direct** forecasting (train to output multiple steps at once); each has trade-offs.


In [ ]:
class LSTMForecaster(nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, batch_first=True)  # 输入1维(单变量), 隐藏hidden / univariate input
        self.fc = nn.Linear(hidden, 1)                    # 隐藏状态 → 下一步预测 / hidden → next-step
    def forward(self, x):
        out, _ = self.lstm(x)                             # out: 每步隐藏状态 / per-step hidden
        return self.fc(out[:, -1])                        # 取最后一步预测下一个 / last step → prediction

torch.manual_seed(0); net = LSTMForecaster(); opt = torch.optim.Adam(net.parameters(), 5e-3)
t0 = time.time()
for epoch in range(300):
    opt.zero_grad(); loss = nn.functional.mse_loss(net(X), Y); loss.backward(); opt.step()
print(f"LSTM 训练完成 ({time.time()-t0:.0f}s), 训练损失 = {loss.item():.4f}")

# 递归多步预测: 用预测值滚动生成后24个月 / recursive multi-step forecast
net.eval(); window = list(s_train[-L:]); preds = []
with torch.no_grad():
    for _ in range(len(test)):
        x = torch.tensor(window[-L:], dtype=torch.float32).view(1, L, 1)
        p = net(x).item(); preds.append(p); window.append(p)   # 预测→追加→继续 / predict, append, continue
lstm_fc = unscale(np.array(preds))
lstm_mape = np.mean(np.abs((test - lstm_fc) / test)) * 100
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(idx[:120], train, label="训练"); ax.plot(idx[120:], test, label="真实", color="green")
ax.plot(idx[120:], lstm_fc, "r--", label="LSTM 递归预测")
ax.legend(); ax.set_title(f"LSTM 递归预测后24个月: MAPE={lstm_mape:.1f}%")
plt.tight_layout(); plt.show()
print(f"LSTM 递归多步预测 MAPE = {lstm_mape:.1f}%")


<a id="4"></a>
## 4. 诚实对比经典方法 + 小结 ⭐ / Honest Comparison with Classical Methods

把 LSTM 和经典方法(Holt-Winters、SARIMA)在**同一个测试集**上比一比。结果可能出乎意料——但这是**重要的实战真相**。
Compare the LSTM against classical methods (Holt-Winters, SARIMA) on the **same test set**. The result may surprise you — but it's an **important real-world truth**.


In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
train_s = pd.Series(train, index=idx[:120])
hw = ExponentialSmoothing(train_s, trend="add", seasonal="mul", seasonal_periods=12).fit()
hw_fc = hw.forecast(len(test)).values
hw_mape = np.mean(np.abs((test - hw_fc)/test))*100
sar = SARIMAX(np.log(train_s), order=(0,1,1), seasonal_order=(0,1,1,12)).fit(disp=False)
sar_fc = np.exp(sar.get_forecast(len(test)).predicted_mean).values
sar_mape = np.mean(np.abs((test - sar_fc)/test))*100

methods = {"Holt-Winters": hw_mape, "SARIMA": sar_mape, "LSTM": lstm_mape}
fig, ax = plt.subplots(figsize=(6.5,4))
bars = ax.bar(methods.keys(), methods.values(), color=["#5a9","#39c","#e76f51"])
for b,v in zip(bars, methods.values()): ax.text(b.get_x()+b.get_width()/2, v+0.2, f"{v:.1f}%", ha="center")
ax.set_ylabel("预测 MAPE (%, 越低越好)"); ax.set_title("同一测试集对比: 小数据上经典方法往往胜过 LSTM")
plt.tight_layout(); plt.show()
for k,v in methods.items(): print(f"  {k:14}: MAPE = {v:.1f}%")
print("\n诚实结论(重要的实战真相): 在这个只有144个点的小数据集上, LSTM 打不过 Holt-Winters/SARIMA!")
print("原因: 深度学习是'数据饥渴'的——需要大量数据才能学好; 小数据上经典统计方法更稳更准")
print("LSTM/深度学习真正的优势场景: ①数据量大 ②多变量(很多相关序列) ③复杂非线性 ④需要跨很多序列共享模型")


```
滑动窗口: 用过去L步→下一步, 在序列上滑动生成监督样本(seq-to-one); 把时序变成机器学习问题
缩放防泄漏: 缩放器只在训练集fit(绝不在含测试的整条序列fit, 否则测试min/max泄漏→评估虚高); 任何统计量只用过去数据
LSTM预测: 读窗口→输出下一步; 多步用递归(预测当输入)但误差累积, 或直接预测(一次出多步)
诚实真相: 小数据(本例144点)上 LSTM 打不过 Holt-Winters/SARIMA! 深度学习数据饥渴
DL优势场景: 数据量大 / 多变量(很多相关序列) / 复杂非线性 / 跨序列共享模型(全局模型)
铁律(再强调): 按时间切+缩放只fit训练集+不打乱 → 防未来信息泄漏
```

### 💡 面试速查 / Interview cheat-sheet
1. **滑动窗口**: 过去L步→下一步, 滑动成监督样本; 把时序变ML问题。
   Sliding window: past L → next, slide into supervised samples.
2. **缩放防泄漏**: 缩放器只在训练集fit(否则测试信息泄漏); 时序头号坑。
   Scaling leakage: fit scaler on train only (else test info leaks); the #1 TS pitfall.
3. **多步预测**: 递归(预测当输入, 误差累积) vs 直接(一次输出多步)。
   Multi-step: recursive (feed predictions, errors compound) vs direct (output many at once).
4. **DL vs 经典**: 小数据经典常更好; DL要大数据/多变量/复杂非线性才值。
   DL vs classical: classical often wins on small data; DL needs scale/multivariate/nonlinearity.
5. **铁律**: 按时间切+不打乱+缩放只fit训练; 防未来泄漏。
   Rules: chronological split + no shuffling + scale on train; prevent future leakage.

### 下一节 / Next
**14.7 时序 CNN / TCN**——除了 RNN, **卷积**也能做时序预测。**TCN(时序卷积网络)** 用**因果卷积**(只看过去) + **空洞卷积**(指数级扩大感受野)处理长序列, 训练比 RNN **更快(可并行)**、长依赖也好。我们会从零理解因果/空洞卷积。
**14.7 Temporal CNN / TCN** — besides RNNs, **convolutions** can forecast too. A **TCN** uses **causal convolutions** (only see the past) + **dilated convolutions** (exponentially growing receptive field) for long sequences, training **faster than RNNs (parallelizable)** with good long-range modeling. We'll understand causal/dilated convolutions from scratch.
